In [0]:
import logging
from pyspark.sql.functions import *

In [0]:
BRONZE_DATABASE = "workspace.default"
SILVER_DATABASE = "workspace.default"

In [0]:
def load_table(table_name):
    return spark.table(
        f"{BRONZE_DATABASE}.{table_name}"
    )

def save_table(df, table_name):

    (
        df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(
            f"{SILVER_DATABASE}.{table_name}"
        )
    )

    print(f"{table_name} saved.")

In [0]:
# Clean Customers
def clean_customers(df):

    print("Cleaning Customers...")

    return (
        df.dropDuplicates(["customer_id"])
          .withColumn(
              "customer_city",
              initcap(trim(col("customer_city")))
          )
          .withColumn(
              "customer_state",
              upper(trim(col("customer_state")))
          )
    )

# Clean Orders

def clean_orders(df):

    print("Cleaning Orders...")

    return (
        df.dropDuplicates(["order_id"])
          .dropna(
              subset=[
                  "order_id",
                  "customer_id",
                  "order_purchase_timestamp"
              ]
          )
          .withColumn(
              "order_status",
              lower(trim(col("order_status")))
          )
    )

# Clean Order Items

def clean_order_items(df):

    print("Cleaning Order Items...")

    return (
        df.dropDuplicates()
          .filter(col("price") > 0)
          .filter(col("freight_value") >= 0)
          .filter(col("order_item_id") > 0)
    )

# Clean Products

def clean_products(df):

    print("Cleaning Products...")

    df = df.dropDuplicates(["product_id"])

    df = df.fillna({
        "product_category_name": "Unknown",
        "product_name_lenght": 0,
        "product_description_lenght": 0,
        "product_photos_qty": 0,
        "product_weight_g": 0,
        "product_length_cm": 0,
        "product_height_cm": 0,
        "product_width_cm": 0
    })

    df = df.withColumn(
        "product_category_name",
        lower(trim(col("product_category_name")))
    )

    df = df.filter(col("product_weight_g") >= 0)

    df = df.filter(col("product_length_cm") >= 0)

    df = df.filter(col("product_height_cm") >= 0)

    df = df.filter(col("product_width_cm") >= 0)

    return df

# Clean Sellers

def clean_sellers(df):

    print("Cleaning Sellers...")

    return (
        df.dropDuplicates(["seller_id"])
          .withColumn(
              "seller_city",
              initcap(trim(col("seller_city")))
          )
          .withColumn(
              "seller_state",
              upper(trim(col("seller_state")))
          )
    )

# Clean Payments

def clean_payments(df):

    print("Cleaning Payments...")

    return (
        df.dropDuplicates()
          .filter(col("payment_value") > 0)
          .filter(col("payment_installments") >= 0)
          .withColumn(
              "payment_type",
              lower(trim(col("payment_type")))
          )
    )

# Clean Reviews

def clean_reviews(df):

    print("Cleaning Reviews...")

    df = (
        df.dropDuplicates()
          .dropna(
              subset=[
                  "review_id",
                  "order_id",
                  "review_score"
              ]
          )
          .fillna({
              "review_comment_title": "",
              "review_comment_message": ""
          })
    )

    df = df.withColumn(
        "review_score",
        col("review_score").try_cast("int")
    )

    df = df.filter(
        col("review_score").between(1, 5)
    )

    df = df.withColumn(
        "review_creation_date",
        try_to_timestamp(col("review_creation_date"))
    )

    df = df.withColumn(
        "review_answer_timestamp",
        try_to_timestamp(col("review_answer_timestamp"))
    )


    return df

# Clean Geolocation

def clean_geolocation(df):

    print("Cleaning Geolocation...")

    return (
        df.dropDuplicates()
          .withColumn(
              "geolocation_city",
              initcap(trim(col("geolocation_city")))
          )
          .withColumn(
              "geolocation_state",
              upper(trim(col("geolocation_state")))
          )
    )

# Clean Category Translation

def clean_category_translation(df):

    print("Cleaning Category Translation...")

    return (
        df.dropDuplicates()
          .withColumn(
              "product_category_name",
              lower(trim(col("product_category_name")))
          )
          .withColumn(
              "product_category_name_english",
              lower(trim(col("product_category_name_english")))
          )
    )

In [0]:
# Load Bronze Tables
customers_df = load_table("bronze_customers")
orders_df = load_table("bronze_orders")
order_items_df = load_table("bronze_order_items")
products_df = load_table("bronze_products")
sellers_df = load_table("bronze_sellers")
payments_df = load_table("bronze_payments")
reviews_df = load_table("bronze_reviews")
geolocation_df = load_table("bronze_geolocation")
category_translation_df = load_table("bronze_category_translation")

# Clean Data
customers_clean = clean_customers(customers_df)
orders_clean = clean_orders(orders_df)
order_items_clean = clean_order_items(order_items_df)
products_clean = clean_products(products_df)
sellers_clean = clean_sellers(sellers_df)
payments_clean = clean_payments(payments_df)
reviews_clean = clean_reviews(reviews_df)
geolocation_clean = clean_geolocation(geolocation_df)
category_translation_clean = clean_category_translation(category_translation_df)

# Save Silver Tables
save_table(customers_clean, "silver_customers")
save_table(orders_clean, "silver_orders")
save_table(order_items_clean, "silver_order_items")
save_table(products_clean, "silver_products")
save_table(sellers_clean, "silver_sellers")
save_table(payments_clean, "silver_payments")
save_table(reviews_clean, "silver_reviews")
save_table(geolocation_clean, "silver_geolocation")
save_table(category_translation_clean, "silver_category_translation")

print("Silver Layer Created Successfully!")

Cleaning Customers...
Cleaning Orders...
Cleaning Order Items...
Cleaning Products...
Cleaning Sellers...
Cleaning Payments...
Cleaning Reviews...
Cleaning Geolocation...
Cleaning Category Translation...
silver_customers saved.
silver_orders saved.
silver_order_items saved.
silver_products saved.
silver_sellers saved.
silver_payments saved.
silver_reviews saved.
silver_geolocation saved.
silver_category_translation saved.
Silver Layer Created Successfully!


In [0]:
display(spark.sql("SHOW TABLES IN workspace.default"))

database,tableName,isTemporary
default,bronze_category_translation,false
default,bronze_customers,false
default,bronze_geolocation,false
default,bronze_order_items,false
default,bronze_orders,false
default,bronze_payments,false
default,bronze_products,false
default,bronze_reviews,false
default,bronze_sellers,false
default,dim_customer,false
